# Healthcare Claims EDA & Cost Analysis
**CMS DE-SynPUF Medicare Claims | 2008-2010 | 230M+ Records**

---

**Objective:** Answer six core business questions a US health plan asks every quarter - where the money goes, for whom, and what's avoidable - using 230 million synthetic Medicare claims processed with DuckDB.

| # | Business Question |
|---|---|
| 1 | What share of total spend comes from the costliest members? |
| 2 | What is the PMPM cost by service line, and how does it trend? |
| 3 | How does chronic-condition layering drive cost? |
| 4 | What is the 30-day readmission rate, and what drives it? |
| 5 | How does cost vary by age and sex? |
| 6 | What are the top cost drivers by DRG, HCPCS, and drug category? |

## 0 - Setup

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# src/ on path
sys.path.insert(0, str(Path.cwd().parent))

from src.config          import *
from src.data_loader     import get_connection, create_views, create_beneficiary_enriched, row_count
from src.quality         import compute_completeness, print_completeness_tiers, check_referential_integrity, check_temporal_consistency
from src.visual_identity import (
    visual_identity, fig_title, adjust_axis, auto_scale_axis,
    fmt_num, fmt_usd, bar_labels, render_table,
    ECON_BG, ECON_RED, NATURE_TEAL, NATURE_CORAL, DEEP_BLUE,
    AMBER, PLUM, WARM_GREY, CHARCOAL, LIGHT_GREY,
    PALETA_SEQ, WHITE
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# DuckDB connection & views─
con = get_connection(memory_limit="6GB", threads=4)
create_views(con)

print("✓ Views created for:", list(DESYNPUF_PATHS.keys()))
print(f"✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4")

✓ Views created for: ['beneficiary', 'inpatient', 'outpatient', 'carrier', 'pde']
✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4


## 1 - Data Quality Assessment

Before any analysis, we validate the data across three dimensions:
1. **Volume** - do row counts match expectations?
2. **Completeness** - what's the fill rate per column?
3. **Referential integrity** - do foreign keys resolve?
4. **Temporal consistency** - are date ranges plausible?

In [32]:
# 1a. Volume check
print("TABLE VOLUME CHECK")
print("=" * 60)
for table, expected in EXPECTED_ROWS.items():
    actual = row_count(con, table)
    status = "✓" if actual == expected else "✗"
    delta  = actual - expected
    print(f"  {status} {table:<14s}  {actual:>14,}  (expected {expected:>14,}, Δ {delta:+,})")

TABLE VOLUME CHECK
  ✓ beneficiary          6,873,274  (expected      6,873,274, Δ +0)
  ✓ carrier             94,863,452  (expected     94,863,452, Δ +0)
  ✓ inpatient            1,332,822  (expected      1,332,822, Δ +0)
  ✓ outpatient          15,826,987  (expected     15,826,987, Δ +0)
  ✓ pde                111,085,969  (expected    111,085,969, Δ +0)


In [33]:
# 1b. Completeness by table
for table in ["beneficiary", "inpatient", "outpatient", "carrier", "pde"]:
    print(f"\n{'─'*60}")
    print(f"COMPLETENESS: {table.upper()}")
    print(f"{'─'*60}")
    df_c = compute_completeness(con, table, exclude_prefixes=[])
    print_completeness_tiers(df_c, max_per_tier=20)


────────────────────────────────────────────────────────────
COMPLETENESS: BENEFICIARY
────────────────────────────────────────────────────────────
Completeness summary (34 columns):

  100% complete: 33 columns
    DESYNPUF_ID                              100.00%
    BENE_BIRTH_DT                            100.00%
    BENE_SEX_IDENT_CD                        100.00%
    BENE_RACE_CD                             100.00%
    BENE_ESRD_IND                            100.00%
    SP_STATE_CODE                            100.00%
    BENE_COUNTY_CD                           100.00%
    BENE_HI_CVRAGE_TOT_MONS                  100.00%
    BENE_SMI_CVRAGE_TOT_MONS                 100.00%
    BENE_HMO_CVRAGE_TOT_MONS                 100.00%
    PLAN_CVRG_MOS_NUM                        100.00%
    SP_ALZHDMTA                              100.00%
    SP_CHF                                   100.00%
    SP_CHRNKIDN                              100.00%
    SP_CNCR                                  

In [34]:
# 1c. Referential integrity
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

ri_checks = [
    ("inpatient",  "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Inpatient → Beneficiary"),
    ("outpatient", "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Outpatient → Beneficiary"),
    ("carrier",    "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Carrier → Beneficiary"),
    ("pde",        "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "PDE → Beneficiary"),
]

for left, right, lk, rk, desc in ri_checks:
    check_referential_integrity(con, left, right, lk, rk, desc)

REFERENTIAL INTEGRITY CHECKS


In [35]:
# 1d. Temporal consistency
print("TEMPORAL CONSISTENCY CHECKS")
print("=" * 60)

tc_checks = [
    ("inpatient",  "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("outpatient", "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("carrier",    "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("pde",        "SRVC_DT",       None,           "2008-01-01", "2010-12-31"),
]

for table, date_from, date_thru, exp_min, exp_max in tc_checks:
    if date_thru:
        check_temporal_consistency(con, table, date_from, date_thru, exp_min, exp_max)
    else:
        # PDE has only a single service date (SRVC_DT).
        # DE-SynPUF stores dates as YYYYMMDD integers (BIGINT), so we
        # cast to VARCHAR then parse to DATE before comparing to date literals.
        date_cast = (
            f"COALESCE(TRY_CAST({date_from} AS DATE), "
            f"TRY_STRPTIME(CAST({date_from} AS VARCHAR), '%Y%m%d')::DATE)"
        )
        result = con.execute(f"""
            SELECT
                MIN({date_cast})::VARCHAR AS min_dt,
                MAX({date_cast})::VARCHAR AS max_dt,
                SUM(CASE WHEN {date_cast} < DATE '{exp_min}'
                           OR {date_cast} > DATE '{exp_max}'
                         THEN 1 ELSE 0 END) AS out_of_range
            FROM {table}
            WHERE {date_from} IS NOT NULL
        """).fetchdf()
        r = result.iloc[0]
        oor = int(r["out_of_range"])
        status = "✓" if oor == 0 else f"✗ ({oor:,} out of range)"
        print(f"  {status}  {table}.{date_from}: [{r['min_dt']} ... {r['max_dt']}]")

TEMPORAL CONSISTENCY CHECKS


  ✓  pde.SRVC_DT: [2008-01-01 ... 2010-12-31]
